## Simple RAG with Memory
Here we can use the Python SDK to develop a RAG agent with Milvus retrievers and Mem0 memory for user preferences.

**Prerequisites:**
- Milvus server running at `localhost:19530`
- Collections named `cuda_docs` and `mcp_docs` with embedded documents
- MEM0_API_KEY environment variable set (for Mem0 cloud) or local Mem0 setup


In [ ]:
import os
import sys

# Import the NeMo-Agent-Toolkit module
module_path = os.path.abspath("../../../src/")
if module_path not in sys.path:
    sys.path.insert(0, module_path)


In [ ]:
import logging

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)


In [ ]:
rag_system_prompt = """Answer the following questions as best you can. \
You may ask the human to use the following tools:

{tools}

IMPORTANT MEMORY TOOL REQUIREMENTS:
1. You MUST call get_memory tool FIRST, before calling any other tools
2. You MUST use user_id "user_12" for all memory operations
3. You MUST include ALL required parameters when calling memory tools
4. When calling add_memory or get_memory, you MUST use the exact format as below, \
don't include any other content, and make sure the input is a valid JSON object.

For get_memory tool, you MUST use this exact format:
{{
    "query": "user preferences",
    "top_k": 1,
    "user_id": "user_12"
}}

For add_memory tool, you MUST use this exact format:
{{
    "conversation": [
        {{
            "role": "user",
            "content": "Hi, I'm Alex. I'm looking for a trip to New York"
        }},
        {{
            "role": "assistant",
            "content": "Hello Alex! I've noted you are looking for a trip to New York."
        }}
    ],
    "user_id": "user_12",
    "metadata": {{
        "key_value_pairs": {{
            "type": "travel",
            "relevance": "high"
        }}
    }},
    "memory": "User is looking for a trip to New York."
}}

You may respond in one of two formats.
Use the following format exactly to ask the human to use a tool:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action (if there is no required input, include "Action Input: None")
Observation: wait for the human to respond with the result from the tool, do not assume the response

... (this Thought/Action/Action Input/Observation can repeat N times. If you do not need \
to use a tool, or after asking the human to use any tools and waiting for the human to \
respond, you might know the final answer.)
Use the following format once you have the final answer:

Thought: I now know the final answer
Final Answer: the final answer to the original input question
"""


In [ ]:
from pydantic import HttpUrl

from nat.agent.sdk import NatReActAgent
from nat.embedder.sdk import NIMEmbedder
from nat.llm.sdk import NimLLM
from nat.plugins.mem0ai.sdk import Mem0Memory
from nat.retriever.sdk import MilvusRetriever
from nat.tool.sdk import AddMemoryTool
from nat.tool.sdk import GetMemoryTool
from nat.tool.sdk import NatRetrieverTool
from nat.utils.sdk.nat_workflow import NatWorkflow

llm = NimLLM(
    model_name="nvdev/meta/llama-3.3-70b-instruct",
    temperature=0,
    max_tokens=4096,
    top_p=1.0,
    name="nim_llm",
)

milvus_embedder = NIMEmbedder(
    model_name="nvidia/nv-embedqa-e5-v5",
    truncate="END",
    name="milvus_embedder",
)

# Create Mem0 memory
memory = Mem0Memory(
    name="saas_memory",
)

cuda_retriever = MilvusRetriever(
    uri=HttpUrl("http://localhost:19530"),
    collection_name="cuda_docs",
    embedder=milvus_embedder,
    top_k=10,
    name="cuda_retriever",
)

mcp_retriever = MilvusRetriever(
    uri=HttpUrl("http://localhost:19530"),
    collection_name="mcp_docs",
    embedder=milvus_embedder,
    top_k=10,
    name="mcp_retriever",
)

cuda_retriever_tool = NatRetrieverTool(
    nat_retriever=cuda_retriever,
    topic="Retrieve documentation for NVIDIA's CUDA library",
    name="cuda_retriever_tool",
)

mcp_retriever_tool = NatRetrieverTool(
    nat_retriever=mcp_retriever,
    topic="Retrieve information about Model Context Protocol (MCP)",
    name="mcp_retriever_tool",
)

add_memory_tool = AddMemoryTool(
    description=(
        "Add any facts about user preferences to long term memory. Always use this if users "
        "mention a preference. The input to this tool should be a string that describes the "
        "user's preference, not the question or answer."
    ),
    nat_memory=memory,
    name="add_memory",
)

get_memory_tool = GetMemoryTool(
    description=(
        "Always call this tool before calling any other tools, even if the user does not mention "
        "to use it. The question should be about user preferences which will help you format your "
        "response. For example: 'How does the user like responses formatted?'"
    ),
    nat_memory=memory,
    name="get_memory",
)

agent = NatReActAgent(
    tools=[cuda_retriever_tool, mcp_retriever_tool, add_memory_tool, get_memory_tool],
    llm=llm,
    verbose=True,
    system_prompt=rag_system_prompt,
)

nat_workflow = NatWorkflow(
    entrypoint=agent,
)


In [ ]:
await nat_workflow.prompt('I prefer concise answers with code examples. How do I install CUDA?')


In [ ]:
import os
from pathlib import Path

path_to_yaml = Path(os.getcwd(), "config", "config_memory.yaml").resolve()

# Create the config directory if it doesn't exist
if not path_to_yaml.parent.exists():
    os.makedirs(path_to_yaml.parent)

# Save the workflow to a config file
nat_workflow.save_to_config_file(path_to_yaml)

# Print out the config file content
with open(path_to_yaml) as f:
    print(f.read())
